### Two Pointers / Sliding Windows

In [4]:
from typing import List, Optional

<b>141.Linked List Cycle </b>. 

fast pointer and slow pointer.  
fast pointer goes two steps at a time. slow pointer goes one step at a time. When they enter the loop, the distance between two pointers will shrink one per time, so finally they will meet.  
快指针每次走两步，慢指针一次走一步。 在慢指针进入环之后，快慢指针之间的距离每次缩小1，所以最终能相遇。

In [2]:
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, x):
#         self.val = x
#         self.next = None

def hasCycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            return True
    return False

<b>142. Linked List Cycle II</b>  
a good explanation [here] (https://leetcode.com/problems/linked-list-cycle-ii/discuss/44781/Concise-O(n)-solution-by-using-C%2B%2B-with-Detailed-Alogrithm-Description) also read the notes in evernotes. 
first step: find where fast pointer and slow pointer meet.
second step: keep where slow pointer stands, move fast pointer to the head, both pointer take one step at a time, the point they meet is the start of the circle => approved mathematically
2 (L1 + L2 ) = (L1 + L2 + n*C) => L1 = (n-1)c + (c-L2).   n is how many time the circle is looped, usually 1
C is distance of the whole loop 
L1: distance between head to start point of loop  
L2: distance between start point to meeting point inside of the loop
distance that slow pointer traveled before they meet: (L1 + L2 )
distance that fast poitner traveled before they meet: (L1 + L2 + n*C)

In [3]:
def detectCycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:       
            fast = head
            while slow!= fast:
                slow = slow.next
                fast = fast.next
            return slow
    return 

<b> 287. Find the Duplicate Number</b>.   
the underlining question is to find if the list contains a loop  
using the value as index for potential chaining.  
index 0 as the head, nums[0] as the next index, and so on    
Example 0=>1=>3=>2=>4=>2.  So 2 is what it should return. Slow initiates at 1, fast initiates at 3, 2 is where the loop begin, 0 is the head

In [4]:
def findDuplicate(nums):   
    if not nums:
        return False
    # ideally initialize slow = fast = head, which is 0 here, but when situation is not allowed, can initiate
    # slow to head.next (which is nums[0]), fast to head.next.next (nums[nums[0]])
    slow = nums[0]      
    fast = nums[nums[0]]
    while slow != fast:
        slow = nums[slow]
        fast = nums[nums[fast]]
    fast = 0    # index 0 is the head, reset fast to the head
    while fast!=slow:
        fast = nums[fast]
        slow = nums[slow]
    return slow   

### way 2: consider the value in the nums as index
(this question has other solutions in other notebook)
if considering the values in the nums list as index, it means there is only one index that is used twice. So can create a new index list as the seats, with len equals to len(nums)-1, then loop over the nums list, if that index is not used, the corresponding position in the new list would have value 0, then take it, if that position already has value 1, that means the index occurred before, so that is the duplicated value in the nums array

this solution does not meet the reqirements of space to O(1)

In [2]:
from typing import List
def findDuplicate(nums: List[int]) -> int:
    index_count = [0] * (len(nums) - 1)
    for i in nums:
        if index_count[i-1] == 0:
            index_count[i-1] += 1
        else:
            return i

<b>80.Remove Duplicates from Sorted Array II </b>

my way: check number k steps way, if current num == num k step away, pop the k step away number out of list, else move current pointer forward,   
Other way: using two pointers, I is one to keep track of where next valid number should be put, also n which is loop the number of the array, compare to I-2 position.

In [3]:
def removeDuplicates(nums):
    
    i=0
    while i < len(nums)-2:  # calculating len(nums) every time instead of saving it as n, because after pop the len has changed
        if nums[i]==nums[i+2]:
            # using pop is not efficient!!
            nums.pop(i+2)
        else:
            i+=1
    return len(nums)
            

In [4]:
nums = [1,1,1,2,2,3]
removeDuplicates(nums)

5

In [5]:
def removeDuplicates(nums):   # better way!
    
    i = 0  # first pointer, index, keep track of NEXT position to put next valid number
    for n in nums:   # second pointer is the actual number of the array
        if (i<2) or (n>nums[i-2]):
            # i < 2 : for the first 2 elements of nums, keep as what they are
            # n > nums[i-2]: if current number > second last number in the valid array, meaning n can be the next valid number
            nums[i] = n
            i += 1
    return i
    

In [6]:
nums = [1,1,1,2,2,3]
removeDuplicates(nums)

5

<b>146.LRU Cache </b>  
Least Recently Used (LRU). 
basic idea
using double linked list structure to keep track of the order, when new item come, always keep it at tail (head), in this way the least used item is always the closest to head (tail) one.  
to implement the double linked list, can do in from scratch, like <b>way 1 </b>  
or do it using python builtin type: OrderedDict in <b> way 2 </b>, in the doc's example, it is actually used in the LRU cases



<b>way 1</b>

In [13]:
class Node:
    """double linked list"""
    def __init__(self, k, v):
        self.key = k
        self.val = v
        self.prev = None
        self.next = None

class LRUCache:

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.head = Node(0, 0)  # can use 0 because assuming valid value is always positive, meaning > 0
        self.tail = Node(0, 0)
        self.head.next = self.tail
        self.tail.prev = self.head
        self.cache = {}  # a dict of double linked node instance
    
    def _remove(self, node):
        """get rid of the linkage will delete the node"""
        prev_node = node.prev
        next_node = node.next
        prev_node.next = next_node
        next_node.prev = prev_node
    
    def _add(self, node):
        """
        add the node to the tail of the double linked list,
        in this way the node near the head will be the least used node, can choose to always add to the head as well, in this way the least used on is closet to the tail
        """
        prev_node = self.tail.prev
        node.prev = prev_node
        node.next = self.tail
        prev_node.next = node
        self.tail.prev = node
        

    def get(self, key: int) -> int:
        if key in self.cache:
            node = self.cache[key]
            self._remove(node)
            self._add(node)  # to make it always at the tail, hence ordered
            return node.val
        return -1
            


    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            # no need to check capacity
            node = self.cache[key]
            self._remove(node)
        node = Node(key, value)
        self._add(node)  # add to the double linked list
        self.cache[key] = node # add to the cache dict
        
        if len(self.cache) > self.capacity:
            # always remove the node closet to head
            del_node = self.head.next
            self._remove(del_node)
            del self.cache[del_node.key]
            
                
# Your LRUCache object will be instantiated and called as such:
# obj = LRUCache(capacity)
# param_1 = obj.get(key)
# obj.put(key,value)

<b> way 2</b>  
OrderedDict: https://docs.python.org/3/library/collections.html#collections.OrderedDict

In [14]:
from collections import OrderedDict
class LRUCache:

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()
        


    def get(self, key: int) -> int:
        if key in self.cache:
            self.cache.move_to_end(key) # to keep its order high up, move item to the end
            return self.cache[key]
        else:
            return -1

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            del self.cache[key]
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)  # FIFO



# Your LRUCache object will be instantiated and called as such:
# obj = LRUCache(capacity)
# param_1 = obj.get(key)
# obj.put(key,value)

<b>42.Trapping Rain Water </b>

two pointers, keep track of left max and right max value. while height[left] and height[right] is current value. when left max <= right max, the water can be trapped is determined by the ledt side, which is left_max - height[left]. vice versa, when left_max > right_max, using the right part to calculat the water amount.

Attention: while left <= right, equal sign will make sure cases like [2, 0, 2] will get chance to calcuate the 2-0 case.

In [15]:
def trap( height):
    """four pointers"""
    left_max, right_max = 0, 0
    left, right = 0, len(height)-1
    result = 0
    while left <= right: # why <=, because case like [2,0,2] wont get chance to calculate the 2-0
        if left_max<=right_max:
            # water amount is dependent on left side
            left_max = max(left_max, height[left])
            result += (left_max - height[left])
            left+=1
        else:
            # water amount is dependent on right side
            right_max = max(right_max, height[right])
            result += (right_max - height[right])
            right -= 1
    return result

In [16]:
height= [ 2, 0, 2]
trap( height)

2

<b>11.Container with Most Water </b>  
somehow similar to the <b>trapping rain water</b> question above. But still different. Trapping water calculates the sum of each hole, so need left_max and right_max to hold the bar also use them to determine where to move.   
This question no need the left_max and right_max, but having the same idea of comparing the current left and right height to determine which direction to move, try to get the max of the container area.  
if left bar < right bar:   
moving right bar wont help to increase the area, since the height of container is determined by the shorter bar, which is left bar, and the x axis length is shrinking because of the moving. So moving the left bar is possible to increase the area because it may pointe to a longer bar that increase the area enougth to offset the shrink brought by smaller x-axis.
if left bar == right bar:
moving to either direction is okay
if left bar > right bar:   
moving the right bar, same reason as above, because moving left bar wont bring any better result, but moving right bar might will





In [41]:
def maxArea(height):
    n = len(height)
    l, r = 0, n-1
    result = 0
    while l < r:
        result =max(result, (r-l)*min(height[l], height[r]))
        if height[l] <= height[r]:
            # moving the r pointer wont help anything, move left may encouter a higher bar that add back the area losed by shorter x-axis length
            # if left height and right height are the same, move either one is fine
            l+=1
        else:
            r-=1
    return result

In [42]:
height = [1,8,6,2,5,4,8,3,7]
maxArea(height)

49

<b>15. 3Sum </b>

This [post](https://leetcode.com/problems/3sum/discuss/232712/Best-Python-Solution-(Explained)) very well explained the O(N^2) solution. It is a two pointers solution. Actually three pointers.  
Sorting takes O(NlogN)  
We iterate through the nums once, and each time we iterate the whole array again by a while loop  
So it is O(NlogN+N^2)~=O(N^2)  

There are some smart thougts here
if starting number > 0, then no need to continue, cause all positive numbers never meet the criteria  
if number in index i is the same as previous number, then continue, to avoid duplicated cases  
when case meet, conduct another 2 while loop to eliminate duplicated left and right pointers value to eliminate duplicated cases

In [22]:
def threeSum(nums):
    nums.sort()
    result = []
    n = len(nums)
    # not need to consider the last two elements as i, because no room for l and r (extra two values)
    for i in range(n-2): 
        if nums[i] > 0: break  # positive numbers wont meet the Q
        # same value no need to consider again, because need to find Unique triplets
        if i > 0 and nums[i]==nums[i-1]: continue
        l = i+1  # left pointer only need to start after i
        r = n-1  # right pointer always start at the end of array
        while l < r:
            total = nums[i]+nums[l] + nums[r]
            if total < 0:
                # too small, should be larger
                l+=1
            elif total > 0:
                # too big
                r-=1
            else:
                result.append([nums[i],nums[l],nums[r]])
                # to abvoid getting repeated result
                while l < r and nums[l]==nums[l+1]: l+=1
                while l < r and nums[r]==nums[r-1]: r-=1
                # suppose we never enter the while loop above, still need to move forward. the l out of the loop still have the same value of the nums[l] before the loop, so still need to move forward
                l+=1
                r-=1
    return result

In [23]:
nums = [-1, 0, 1, 2, -1, -4]
threeSum(nums)

[[-1, -1, 2], [-1, 0, 1]]

### some similar questions to 3Sum  
<b>1.two sum </b>

Array is not sorted, and sorting in advance wont help either because it wants to return the oringial index. so cant use two pointers. Using hashMap to store the value as key and its index as val, and loop once can solve the prolbme.  
Time complexity: O(N)

In [24]:
def twoSum(nums, target):
    """
    two pointers
    is the array sorted?
    """

    hash_map={}
    for ind, n in enumerate(nums):
        val = target - n
        if val in hash_map:
            return [hash_map[val], ind]
        hash_map[n]=ind
    # not returning False because the Q assume there alwasy exist an unique answer

In [25]:
nums = [2, 7, 11, 15]
target = 9

twoSum(nums, target)

[0, 1]

<b>167.Two Sum II - Input array is sorted </b>

using two pointers. start from left and right, moving to the right direction  
this is what is used in the <b>3 Sum question</b> too

In [26]:
def twoSum(numbers, target):
    i, j = 0, len(numbers)-1
    while i < j:
        val = numbers[i] + numbers[j]
        if val  < target:
            i+=1
        elif val > target:
            j-=1
        else:
            return [i+1, j+1]

In [32]:
numbers = [2,7,11,15] 
target = 9
twoSum(numbers, target)

[1, 2]

<b> 170 Two Sum III - data structure design </b>

re-use the hashmap idea in 2Sum problem, here using a counter map because the data from stream can be duplicated

In [2]:
from collections import defaultdict
class TwoSum:

    def __init__(self):
        self.num_counter = defaultdict(int)

    def add(self, number: int) -> None:
        self.num_counter[number] += 1

    def find(self, value: int) -> bool:
        for key, count in self.num_counter.items():
            if value == 2*key and self.num_counter[key]>1:
                return True
            elif value != 2*key and value - key in self.num_counter:
                return True
        return False
        


# Your TwoSum object will be instantiated and called as such:
# obj = TwoSum()
# obj.add(number)
# param_2 = obj.find(value)

<b> 653 Two Sum IV - input is a BST </b>

transform the BST into a sorted list, then using the 2 pointers way to validate if there are such two value

In [6]:
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
class Solution:
    def findTarget(self, root: Optional[TreeNode], k: int) -> bool:
        # inorder traversl to get a sorted list
        def inorder(node, sorted_nodes):
            if not node: return
            inorder(node.left, sorted_nodes)
            sorted_nodes.append(node.val)
            inorder(node.right, sorted_nodes)
        sorted_nodes = []
        inorder(root, sorted_nodes)
        l, r = 0, len(sorted_nodes)-1
        while l < r:
            val = sorted_nodes[l]+sorted_nodes[r] 
            if val == k: 
                return True
            elif val < k:
                l+=1
            else:
                r-=1
        return False
        

<b>16. 3sum closest </b>  
same as 3sum question, first need to sort the array. 3 pointers, move l and r based on sum and target  
but the actual returned value is min distance  
which is also O(N^2)

In [34]:
def threeSumClosest(nums, target):
    n = len(nums)
    result = float("inf")
    nums.sort()
    for i in range(n-2):
        # assume only have exactly one solution
        l = i+1
        r = n-1
        while l < r:
            val = nums[i]+nums[l]+nums[r]
            if val > target:
                r-=1                 
            elif val < target:
                l+=1
            else:
                return val
            if abs(result-target) >  abs(val-target):
                result = val
    return result

In [35]:
nums = [-1,2,1,-4]
target = 1
threeSumClosest(nums, target)

2

<b>259. 3Sum Smaller </b>

Given an array of n integers nums and an integer target, find the number of index triplets i, j, k with 0 <= i < j < k < n that satisfy the condition nums[i] + nums[j] + nums[k] < target.

key at how to update the result count.  
when fix i, and find a pair of [k, j] that satisfy the constraint, because the array is sorted, any pair of [i, k, (k+1 to j)]: i, k fixed, j can choose from k+1 until j will be a valid solution, the number is j-k.  

when k, or clled l below, moves, the pair of i, k is different, then the i,k, all those j pair are diff  

when j, or called r below, moves, it is helping to locate the valid pairs, the pair only counts when nums[i]+nums[l]+nums[r]<target


In [36]:
def threeSumSmaller(nums, target):
    """tricky"""
    nums.sort()  # okay to sort, the i<j<k just mean no duplicate usage, and the answer just count
    n = len(nums)
#     if n <=2:
#         return 0
    result = 0
    for i in range(n-2):
        l = i+1
        r = n-1
        while l < r:
            if nums[i]+nums[l]+nums[r]<target:
                result += (r-l)  # key point, otherwise will miss cases [i, l, between l and r], since r only updates when sum >= target
                l +=1
            else:
                r -=1
    return result

In [37]:
nums = [-2,0,1,3]
target = 2
threeSumSmaller(nums, target)

2

<b>18. 4sum </b>

find index tuple that s[i] + s[j] + s[k]  +s[q] == target, and the result index tuple must be unique
recursive, create a helper function to solve the 2 sum problems by using two pointers against a sorted list

then recursively reduce N problems 2 sum problems.
time complexity:
-   level = 4: loop over all items O(N)
-   recursive down 1 level, level 3: loop over all other items O(N-1)
-   now level 2, trigger the 2sum function, takes roughly O(N)
-   sorting takes O(nlogN)
-   so total O(NlogN + N**3)
space complexity:
-   recursive depth depends on the how many sum: M sum then M-2 depth
-   each recursive takes O(N) for tmp result -> O(4*N) => O(N)
-   results storage: suppose Q different combination, O(4Q), Q is proportional to C(n,4) as worse case, any 4 out of the N number is a valid one -> O(N**4)

In [6]:
class Solution:
    def fourSum(self, nums: List[int], target: int) -> List[List[int]]:
        """recursive, utilizing the function of 2 sum, can generalize to find sum of N items"""
        def find_sum(l, r, target, N, current_result, results):
            """
            l: left index of the nums array
            r: right index of the nums array
            N: size
            current_result: a list, containing previous needed items from the nums array to construct a valid result
            results: a list of list, the final result
            """
            # base case 1: if length of nums slice < N, or target value cant be satisfied with the current nums slice
            if r-l+1 < N or N < 2 or target < nums[l]*N or target > nums[r]*N:
                return
            if N == 2:
                # reuse the 2 sum functionality
                while l < r:
                    val = nums[l] + nums[r]
                    if val == target:
                        # find it!
                        results.append(current_result + [nums[l], nums[r]])
                        # to avoid duplicates
                        while l < r and nums[l] == nums[l+1]: l+=1
                        while l < r and nums[r] == nums[r-1]: r-=1
                        l+=1
                        r-=1
                    elif val < target:
                        l += 1
                    else:
                        r -= 1
            else:
                # recursively reduce N
                for i in range(l, r+1):
                    # to avoid duplicates 
                    if i == l or (i>l and nums[i-1]!=nums[i]):
                        find_sum(i+1, r, target-nums[i], N-1, current_result+[nums[i]], results)
        nums.sort()
        results = []
        find_sum(0, len(nums)-1, target, 4, [], results)
        return results
                        


<b> 454 4Sum II</b>

return typle of index, no need to worry about duplicated actual value because index combination will always be unique

same as 4sum, but no need to check duplicate, and append index instead of actual value to the result list

In [ ]:
from collections import Counter
class Solution:
    def fourSumCount(self, nums1: List[int], nums2: List[int], nums3: List[int], nums4: List[int]) -> int:
        # greedy: N^4 different combinations
        # but can reduce to N^2 by calculating the sum of two list first by looping over
        # then calculating the sum of rest two lists by looping over
        # using a map to store the {val: count}, if the second sum has value -val.
        # then we found one valid tuple

        AB = Counter(a + b for a in nums1 for b in nums2)
        # a + b + c + d = 0
        return sum(AB[-(c+d)] for c in nums3 for d in nums4)



    
        

<b>238.Product of Array Except Self </b>  
Actually not two pointers, just loop  
Without using division, to complete O(N) that means can only loop the whole array constant times.  
Can construct a left array, stores the left product of each num in the array  
And another loop to construct a right array, stores the right product of each num in the array  
Then result of left * right for each position in the array.   
To optimize the space, we can use result array to store the left array at the first loop, the in second loop, construct var right, and directly apply it to the left array, which is result array, and update the result array on the fly.
a very good explanation in the first comment of this [post](https://leetcode.com/problems/product-of-array-except-self/discuss/65622/Simple-Java-solution-in-O(n)-without-extra-space)

a version with explanations of having the left and right array

In [46]:
def productExceptSelf(nums):
    n = len(nums)
    result, left, right = [0]*n, [0]*n, [0]*n
    # first num wont have left product, so default to 1, that means result[0]=1
    left[0] = 1   
    for i in range(1, n):
        # the first loop, construct left product for each n in nums
        left[i] = left[i-1] * nums[i-1]
    # last num wont have right product, so default to 1
    right[-1] = 1
    for i in range(n-2, -1, -1):
        # starting from back, construct the right product, and fill the result at one loop
        right[i] = right[i+1] * nums[i+1]
    for i in range(n):
        result[i] = left[i]*right[i]
    return result

In [47]:
nums = [1,2,3,4]
productExceptSelf(nums)

[24, 12, 8, 6]

this version is more concise from the above => same idea as above

In [51]:
from collections import Counter
def productExceptSelf(nums):
    n = len(nums)
    result = [0]*n
    # first num wont have left product, so default to 1
    result[0] = 1   
    for i in range(1, n):
        # the first loop, construct left product for each n in nums
        result[i] = result[i-1]*nums[i-1]
    # last num wont have right product, so default to 1
    right = 1
    for i in range(n-1, -1, -1):
        # starting from back, construct the right product, and fill the result at one loop
        # result = left prodct * right product, and we use result list to store the left product above
        result[i] *= right
        right *= nums[i]  # this is the right product for next value in nums
    return result

<b>76.Minimum Window Substring <b>

Sliding window (two pointers), first using right pointer to find spot where left and right together construct a window that includes all chars needed. Then move the left window to find the correct start to get rid of some unnecessary chars. Two sliders keep moving forward. Worst case left and right pointers all visited the whole array, but still time complexity is O(N)  
1. Use two pointers: start and end to represent a window.  
2. Move end to find a valid window.  
3. When a valid window is found, move start to find a smaller window.  

Note: this solution is not too intuitive, check the second solution

why start index j from 1?  
when set j to starting from 0, if the test case is s="a", t="aa", supposed to return empty string, but if set j to 0, the return would be s[start: end+1], which returns "a". Because end is from j, and it is a needed index, so need to +1 to ensure j is included in the result. But the s[start:end+1] wont work for the above test case. that means if using j start from 0, need special care of the cases before.   
if set j to start from 1, result is s[start:end], the above case would be s[0:0] which returns empty string, no need special care

In [48]:
def minWindow(s, t):
    """smart sliding window"""
    need = Counter(t)  # dict to see what chars should include. Magic about Counter dic is even c not in the dict, can still call dict[c] which will give 0 as the count
    missing = len(t) # how many chars are still needed
    # two pointers that save the smallest window found so far
    start, end = 0, 0 
    i = 0   # another pair of pointers i, j used to do sliding in each step of loop
    for j, char in enumerate(s, 1):  # start index j from 1
        if need[char] > 0:
            # find one needed char
            missing -=1
        # even char not in t, the counter dict will by default have it as count 0
        need[char] -= 1

        # if j slides to a position where all chars are found, move the i to update the window size, smaller but still holds what you need
        if missing == 0:
            # j starts from 0, so i<=j
            # a needed char starts count > 0, an unneeded char starts from 0
            # so when loop using i from start, if it is an unneeded char, the number is flipping back and i hits j, if it is a needed char before i hits j there is one splace need == 0
            while i < j and need[s[i]]<0:
                need[s[i]] +=1
                i += 1
            # now i points to a new start of a small sliding window that still holds every chars needed
            if end== 0 or j-i < end - start: 
                start, end = i, j # note, here end and j are the last needed char
            # now i need to move to the next position, the current position is a needed char, before moving, update the need and missing
            need[s[i]] +=1
            missing += 1
            i += 1
    return s[start: end]

In [52]:
s="a"
t="aa"
minWindow(s, t)

''

In [53]:
s = "ADOBECODEBANC"
t = "ABC"
minWindow(s, t)

'BANC'

another way, maybe easier for understanding

time complexity O(2*N) = O(N), i and j only loop the array once

In [ ]:
from collections import math
class Solution:
    def minWindow(self, s: str, t: str) -> str:
        counter = Counter(t)
        missing = len(t) # keep track of sum of missing char, can do sum(counter)<=0 as replacement but increase time complexity
        i = 0
        res = ""
        min_l = math.inf
        for j, c in enumerate(s):
            if counter[c] > 0:
                # meet one required char
                missing -= 1
            # only update char same in t, other char does not need to worry about
            # the counter[c] can be negative, which is important, that means there are extra char met along
            # the way, so i can update accordingly.
            if c in counter:
                counter[c] -= 1
            # update the min_l and res only when all chars are found
            # then move i forward to shrink the window
            while missing == 0 and i <= j:
                if j-i+1 < min_l:
                    min_l = j-i+1
                    res = s[i: j+1]
                if s[i] in counter:
                    # if it is a negative value, we know there were something in the current window not used up yet
                    counter[s[i]] += 1
                    if counter[s[i]] > 0:
                        missing += 1
                i += 1
        return res

<b> 763. Partition Labels </b>
actually three pointers.  
first construct a hashmap to store the right most index for each unique character
two pointers: start and end to mark the window of each partition
another pointer while loop the string as array, update the last index of the char by current pointer.
the `if i == end` is a smart step to find the right start

why last would be updated while looping?
for example: between those two occurrences of 'a', there could be other labels that make the minimum size of this partition bigger. 

time complexity O(N)


In [54]:
def partitionLabels(S):
    """three pointers actually"""
    # save the right most index of each char
    hashmap = {c: i for i, c in enumerate(S)}
    result = []
    # two pointers here
    start,end = 0, 0

    # construct another pointer
    for i, c in enumerate(S):
        end = max(end, hashmap[c])
        if i == end:
            result.append(end-start+1)
            start = i+1

    return result

In [55]:
S = "ababcbacadefegdehijhklij"
partitionLabels(S)

[9, 7, 8]

<b> 253.Meeting Rooms II </b>

In [59]:
def minMeetingRooms(intervals):
    starts = sorted(i[0] for i in intervals)
    ends = sorted(i[1] for i in intervals)
    available = room_needed = 0
    s = 0 # start meeting pointer
    e = 0  # end meeting pointer
    while s < len(intervals):
        # need a room
        if starts[s] < ends[e]:
            if available > 0:
                available -= 1
            else:
                room_needed += 1
            s+=1
        else:
            # a previous meeting is ended before the next one starts:
            # it will keep checking, and it works becauses when s == e, it will need a room
            available += 1
            e+=1
    return room_needed

In [60]:
intervals = [[0,30],[5,10],[15,20]]
minMeetingRooms(intervals)

2

In [61]:

intervals=[[0,2],[1,3],[2,4],[3,5],[4,6]]
minMeetingRooms(intervals)

2

<b>435.Non-overlapping Intervals</b>

sort by end is the key, keep the intervals ends first, so latter ones can have more room, so will reduce less amount of items compare with sort by start.

In [62]:
def eraseOverlapIntervals(intervals) :
    """find max number of items that are not overlapping"""
    if not intervals:
        return 0
     # sort by last number, so can find MIN number of intervals need to remove
        # keep the one ends early will give latter one more room to not overlap
    intervals.sort(key = lambda x: x[1]) 
    start = intervals[0]
    count = 1 # count is the non-overlapping count
    for end in intervals[1:]:
        if start[-1]<=end[0]:
            count += 1
            start = end
    return len(intervals)-count

In [63]:
intervals = [[1,2],[2,3],[3,4],[1,3]]
eraseOverlapIntervals(intervals)

1

another way, may be easier to understand

In [64]:
def eraseOverlapIntervals(intervals):
    """find max number of items that are not overlapping"""
    if not intervals:
        return 0
    # sort by last number, so can find MIN number of intervals need to remove

    # keep the one ends early will give latter one more room to not overlap
    intervals.sort(key = lambda x: x[1]) 
    end = float('-inf')
    result = 0
    for i in intervals:
        if i[0]>=end:
            # no overlapping, keep i
            end = i[1]
        else:
            # overlapping occurs
            result+=1
    return result

In [65]:
intervals = [[1,2],[2,3],[3,4],[1,3]]
eraseOverlapIntervals(intervals)

1

<b> 53. Maximum Subarray </b>

greedy way:  
current element = [-2, <b>1</b>, -3, <b>4</b>, -1, 2, 1, -5, 4]  
current max sum = [-2, <b>1</b>, -2, <b>4</b>, 3, 5, 6, 1, 5] bold num is where curr max restart  
max sum so far  = [-2, 1, 1, 4, 4, 5, 6, 6, 6]. => 6

loop the array once, so O(N)

In [ ]:
def maxSubArray(nums):
    curr_sum = max_sum = nums[0]
    for n in nums[1:]:
        curr_sum = max(curr_sum+n, n) # check where the sub array should restarts
        # restart the subarray at a better position, so curr_sum helps determine where it should start
        max_sum = max(max_sum, curr_sum)
        # control where the subarray should stop
    return max_sum

<b> 48 Rotate image </b>

way 1: first transpose (a[i][j] = a[j][i]). then reverse each row

In [1]:
def rotate(matrix):
    """
    Do not return anything, modify matrix in-place instead.
    """
    # transpose first, then reverse each row
    # transpose is litteraly a[i][j] = a[j][i]
    # imagine item in the diagonal line is steady, left and right of the diagonal line swapped. 
    # then remember to reverse each row to get the 90 degree rotation effect

    n = len(matrix)
    for i in range(n):
        for j in range(i, n):
            # start from the first column, never go back
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]

    for i in range(n):
        matrix[i]=matrix[i][::-1]

    return matrix

In [2]:
matrix = [[1,2,3],
  [4,5,6],
  [7,8,9]]
rotate(matrix)

[[7, 4, 1], [8, 5, 2], [9, 6, 3]]

<b> 209 minimum size subarray sum </b>

[link](https://leetcode.com/problems/minimum-size-subarray-sum/solutions/59090/c-o-n-and-o-nlogn/)
there are different interpretation of this functions

first way: if the order of subarray does not matter, meaning the items of the original array can be re-arranged.

O(NlogN) because sorting the array first

for example: [4, 2, 4], target 8. if the item can be re-ordered, the shortest subarray should be [4, 4] so length is 2, but if the item of original array is fixed, then the minimum subarray is [4, 2, 4] 

another test case [12,28,83,4,25,26,25,2,25,25,25,12], target=213, my solution gives the minimum length of subarray as 7, but the result said it is 8 which is not the case.

In [2]:
from typing import List
def minSubArrayLen( target: int, nums: List[int]) -> int:
    # O(NlogN) because of sorting
    if sum(nums) < target:
        return 0

    nums.sort(reverse=True)
    sub_sum = 0
    for i in range(len(nums)):
        sub_sum += nums[i]
        if sub_sum >= target:
            return i+1

    return 0    

second way: if the items in original array is fixed, so "subarray" just slicing the original array

complexity: O(N), using two pointers.
set l, r two pointers, r keeps moving forward until sum s>=target, then keep track of min of the subarray length, then move l forward until sum s<target. repeat the process until r reaches the end of the array.

In [8]:

def minSubArrayLen(target: int, nums: List[int]) -> int:
    # O(N) since the items in nums will be visited constant times by r and l two pointers
    l=0
    r=0
    min_len = float('inf')
    s = 0
    if sum(nums) < target: return 0
    while r < len(nums):
        s += nums[r]
        r+=1
        while s>=target and l<=r:
            min_len = min(min_len, r-l)
            s -= nums[l]
            l+=1
    return min_len


<b> 54. spiral matrix </b>

giving a matrix, list the element by clock wise order

way one: using i, j to locate the position, using dx, dy. to control the direction change of clock wise order

- (0, 1) moving right, 
- (1, 0) moving down,
- (0, -1) moving left,
- (-1, 0) moving up

In [7]:
def spiralOrder( matrix: List[List[int]]) -> List[int]:
    m = len(matrix)
    n = len(matrix[0])

    # (0, 1) moving right, 
    # (1, 0) moving down,
    # (0, -1) moving left,
    # (-1, 0) moving up
    idx, idy = 0, 1
    i, j = 0, 0
    output = []
    visited = [[0]*n for _ in range(m)]
    for _ in range(m*n):
        visited[i][j] = 1
        output.append(matrix[i][j])
        # check if next step needs to change direction
        if visited[(i+idx)%m][(j+idy)%n]:
            # visited 
            idx, idy = idy, -idx
        i += idx
        j += idy
    return output


way 2, construct the row, col range for each direction change.


In [9]:
def spiralOrder( matrix: List[List[int]]) -> List[int]:
    result = []
    if not matrix or len(matrix[0])==0:
        return result
    n = len(matrix)
    m = len(matrix[0])
    row_begin, row_end = 0, n-1
    col_begin, col_end = 0, m-1
    while row_begin <=row_end and col_begin <= col_end:
        # traverse right
        for j in range(col_begin, col_end+1):
            result.append(matrix[row_begin][j])
        row_begin+=1
        # traverse down
        for i in range(row_begin, row_end+1):
            result.append(matrix[i][col_end])
        col_end -=1
        if row_begin <= row_end:
            # traverse left
            for j in range(col_end, col_begin-1, -1):
                result.append(matrix[row_end][j])
        row_end -= 1

        if col_begin <= col_end:
            # traverse up
            for i in range(row_end, row_begin-1, -1):
                result.append(matrix[i][col_begin])
        col_begin+=1
    return result





<b> 59 Spiral matrix II </b>

giving an integer n, fill the matrix in size n*n with value 1 to n*n in clock wise order

way one can use the similar solution with <b>54</b>

In [8]:
def generateMatrix(self, n: int) -> List[List[int]]:
    """utilize the feature that matrix size is N^2"""
    matrix = [[0]*n for _ in range(n)]
    i, j = 0, 0
    di, dj = 0, 1 # control the moving direction, start with moving to the right, then 1,0 (down), 0,-1 (left), -1,0 (up)
    for k in range(n*n):
        matrix[i][j] = k+1
        if matrix[(i+di)%n][(j+dj)%n]:
            # if that position already has non-0 value, it is already visited, need to change direction
            di, dj = dj, -di # clock wise
        i += di
        j += dj
    return matrix

way 2, go over the four direction in clock wise one by one 

In [ ]:
class Solution:
    def generateMatrix(self, n: int) -> List[List[int]]:
        begin_row, end_row = 0, n-1
        begin_col, end_col = 0, n-1
        matrix = [[1]*n for _ in range(n)]
        num = 1
        while begin_row<=end_row and begin_col <= end_col:
            # move right, row stays same, col changes
            for j in range(begin_col, end_col+1):
                matrix[begin_row][j] = num
                num += 1
            begin_row += 1
            # move down, col stays same, row changes
            for i in range(begin_row, end_row+1):
                matrix[i][end_col] = num
                num+=1
            end_col-=1
            
            # move left, row stays same, col changes
            if begin_row<end_row:
                for j in range(end_col, begin_col-1, -1):
                    matrix[end_row][j] = num
                    num+=1
                end_row-=1
            
            # move up, col stays same, row changes
            if begin_col < end_col:
                for i in range(end_row, begin_row-1, -1):
                    matrix[i][begin_col] = num
                    num+=1
                begin_col+=1
        return matrix
                    

<b> 2863 maximum length of semi-decreasing subarrays </b>

You are given an integer array nums.

Return the length of the longest semi-decreasing subarray of nums, and 0 if there are no such subarrays.

A subarray is a contiguous non-empty sequence of elements within an array.
A non-empty array is semi-decreasing if its first element is strictly greater than its last element.
 

Example 1:

Input: nums = [7,6,5,4,3,2,1,6,10,11]  
Output: 8  
Explanation: Take the subarray [7,6,5,4,3,2,1,6].  
The first element is 7 and the last one is 6 so the condition is met.  
Hence, the answer would be the length of the subarray or 8.  
It can be shown that there aren't any subarrays with the given condition with a length greater than 8.  

Example 2:

[-715,-393,486,-895,737,-92,-354]  
output: 5

i started with a solution that exceeds the time limit


In [1]:
 def maxSubarrayLength(nums):
    """
    :type nums: List[int]
    :rtype: int
    """
    # O(N*N)
    # pointer A start from begginning, pointer B start from end end until find the first value < what A points at

    l = len(nums)
    max_len = 0
    for i in range(l):
        for j in range(l-1, -1, -1):
            if nums[i] > nums[j]:
                max_len = max(max_len, j-i+1)
                break
    return max_len


then optimize from there 

In [2]:

def maxSubarrayLength(nums):
    """
    :type nums: List[int]
    :rtype: int
    """
    # should not be O(N*N) anymore, but not sure if it is O(N)
    # pointer A start from begginning, pointer B start from end end until find the first value < what A points at

    l = len(nums)
    max_len = 0
    for i in range(l):
        if i!=0 and nums[i] <= nums[i-1]:
            # skip nums[i] since it wont get longer than nums[i-1]
            continue
        # if rest of cases wont get larger result, no need to compare anymore
        if (l-i) < max_len:
            return max_len
        # j can stop when meet i
        for j in range(l-1, i, -1):
            if nums[i] > nums[j]:
                max_len = max(max_len, j-i+1)
                break
    return max_len



this problem can be solved by monotonic stack in O(N), but i still think mine is easier for interpretation and the final time is not much difference

In [3]:

def maxSubarrayLength( nums):
    """
    :type nums: List[int]
    :rtype: int
    """
    # O(N)
    # stack stores values from 1st item of nums that keeps decreasing
    # the first item determines the bar of what is saved to the stack
    # so the stack stores an increasing array

    l = len(nums)
    max_len = 0
    stack = []
    for i in range(l):
        if (not stack) or nums[i] > nums[stack[-1]]:
            # the stack actually stores index
            stack.append(i)

    # loop from back to top
    for j in range(l-1, -1, -1):
        while stack and nums[stack[-1]] > nums[j]:
            # then a subarray from stack[-1] to i is found
            i = stack.pop()
            # the start of subarray will moving backward to increase the max_len after poping out value from stack
            max_len = max(max_len, j-i+1)

    return max_len



<b> 1101 the eariliest time when everyone becomes friends </b>

using union find.  
find method: find the final parent  
union: if two does not have same parent, then connect them  
the group size is essential, initial value is N meaning there are separate N group, whenever two ppl are connected by union method, the group size reduced. When group size is 1 that means everyone is in same group, then time found (here should sort the original list by timestamp)

In [5]:

def earliestAcq(logs, n):
    """
    :type logs: List[List[int]]
    :type n: int
    :rtype: int
    """
    # sort the logs by timestamps
    # union element who has same parent
    # whe there is only one parent, it is end

    self.group_size = n # intially every friend is separated, so n group
    self.groups = {i: i for i in range(n)}

    def find(x):
        if self.groups[x]!=x:
            # x:x means finding the parent, to the end
            # if not means x has parent, keep moving up and update x's parent with grand parent
            self.groups[x] = find(self.groups[x])
        return self.groups[x]
    def union(x, y):
        x, y = find(x), find(y)
        if (x != y):
            # separate group size should reduce
            self.group_size -= 1
            # connect x and y
            self.groups[x] = y
    # sort by timestamps
    for t, x, y in sorted(logs):
        union(x, y)
        if self.group_size == 1:
            # only one group left, and since logs is already sorted, the time should be ealiest
            return t
    return -1

<b>. 1712 ways to split arrays into 3 subarrays </b> 

way 1: initial solution, easily cause exceed time limit error

In [6]:
from typing import List
def waysToSplit(nums: List[int]) -> int:
    res = 0
    i = 0
    n = len(nums)
    total = sum(nums)
    left, right = 0, 0
    while i < n:
        left += nums[i]
        j = i + 1
        mid = 0
        while j < n:
            mid += nums[j]
            if mid >= left:
                right = total - mid - left
                if right >= mid:
                    res += 1
                    j += 1
                else:
                    break
            else:
                j+=1
        if left > total-left:
            break
        i += 1
    return res



way 2: first fix i, so array [0,i] is the left, then find j, k, where [i+1, j] start to be larger then left, and [i+1, k] just cut the edge where one more element will make right smaller then mid. so there are k-j ways for fixed i to find the cutting solutions. Then move the i forward, j and k will keep moving forward and neve restart so whole process is quicker

time complexity O(N) i, j, k only loop n once.  
space complexity O(1) since reuse the array nums

In [7]:

def waysToSplit(nums: List[int]) -> int:
    res = 0
    j, k = 0, 0
    n = len(nums)
    for i in range(1, n):
        nums[i] += nums[i-1] # calculate the cumulative sum

    for i in range(n - 2):
        # n-1, n-2 last two index, for the mid and right
        # j is the first valid index to make mid > left
        while j <= i or ( j < n-1 and nums[j] - nums[i] < nums[i]):
            # nums[j] - nums[i] -> mid, nums[i]: left
            j += 1
        # k is the first invalid index that will make right < mid
        while k < j or (k < n-1 and nums[k] - nums[i] <= nums[-1] - nums[k]):
            k += 1 
        res =( res + k-j ) % (10**9+7)
        # j, k keep moving forward, not reset, so whole process is quicker
    return res



using binary search time complexity O(NlogN)

In [8]:
from bisect import bisect_left, bisect

def waysToSplit(nums: List[int]) -> int:
    res = 0
    n = len(nums)
    for i in range(1, n):
        nums[i] += nums[i-1] # calculate the cumulative sum

    for i in range(n - 2):
    # n-1, n-2 last two index, for the mid and right
        # find j, the first valid index to make mid >= left
        # mid = nums[j] - nums[i], left: nums[i] => nums[j] - nums[i] >= nums[i]
        # => nums[j] >= 2*nums[i]
        j = bisect_left(nums, 2*nums[i], i+1) # i+1 is the starting point
        # find k, the first invalid index that will make right < mid
        # right = nums[-1] - nums[k], mid = nums[k] - nums[i]
        # right < mid => nums[-1] - nums[k] < nums[k] - nums[i]
        # => nums[-1] + nums[i] < 2 * nums[k] => nums[k] > (nums[-1] + nums[i])/2
        k = bisect(nums, (nums[-1] + nums[i])/2)
        # check if k, j is valid, largest k is n-1, bisect right could set value to n which is the next item of the last item
        res += max(0, min(n-1, k) - j)

    return res % (10**9+7)




<b> 503 next greater elements </b>

loop the list twice so to find the next item in the connected circle.  
using a prev_index_stack to save previously visited index, so in the for loop, if the item now is greater than the previously visited ind, then the next higher val for this prev visited ind is found.



In [9]:
from typing import List
def nextGreaterElements(nums: List[int]) -> List[int]:
    """
    time complexity: main for loop is O(2N) = O(N), the prev_ind_stack only has max len
    N, since each element can only be pushed and popped once. so this while loop is O(N) 
    in total across all for loop iteration. 
    so total time complexity is O(N) + O(N) = O(N)
    """
    n = len(nums)
    res = [-1]*n
    prev_ind_stack = []

    # loop over the nums list twice is enough to generate the results
    for i in range(2*n):
        val = nums[i%n]

        while prev_ind_stack and nums[prev_ind_stack[-1]] < val:
            # current value is greater than the previous visited val at index stored in the stack
            # that means the previous index find its next higher value
            # keep popping the ind from the stack, because current val can be the next higher value for multiple previous index
            res[prev_ind_stack.pop()] = val

        # a stack to keep previously visited index
        if i < n:
            prev_ind_stack.append(i)
    return res


<b> 881 boats to save people </b>

my first intuition is to sort the people's weight, add to the boat first to people with largest weight (at the right most position), then check if the people with lightest weight (left most) can be added to the boat or not. Since the boat can only take at most 2 people, increment the boat number accordingly.

step 1: using sort, then deque, then while loop, time complexity is 
- sort: O(NlogN), loop over the people list, O(N), total is O(N+NlogN)

In [5]:
from collections import deque

def numRescueBoats(people: List[int], limit: int) -> int:
    """
    one boat at most two people
    """
    people.sort()
    queue = deque(people)
    res = 0
    while queue:
        current_limit = limit
        # according to the problem people[i] always <= limit
        # everytime pop the largest value to the boat
        current_limit -= queue.pop()
        # check if additional 1 person can be added to the boat
        if queue and queue[0] <= current_limit:
            queue.popleft()
        res += 1
    return res

In [6]:
people = [2,49,10,7,11,41,47,2,22,6,13,12,33,18,10,26,2,6,50,10]
numRescueBoats(people, 50)

11

step 2, using two pointers is much simpler

In [7]:

def numRescueBoats(people: List[int], limit: int) -> int:
    """
    one boat at most two people, 
    after sort the poeple list using two pointers to loop over
    """
    people.sort()
    left = 0
    right = len(people) - 1
    res = 0
    while left <= right:
        if people[left] + people[right] <= limit:
            right -= 1
            left += 1
        else:
            right -= 1
        res += 1
    return res

<b> count subarrays with fixed bounds </b>

smart ways to use 3 pointers and sliding window. the solution only loop over the array once, so time complexity is O(N), and the space complexity is only O(1)

In [8]:

def countSubarrays(nums: List[int], minK: int, maxK: int) -> int:
    """
    three pointers
    - jbad: most recent index that does not satisfy constraint of minK <= elements <= maxK
    - jmin: most recent index to have nums[jmin] == minK
    = jmax: most recent index to have nums[jmax] == maxK
    loop over the nums, for each position i, update the three index. The valid starting position of the valid substring is 
    within the range of [jbad+1, min(jmin, jmax)], anything between this range can be a valid starting point.
    ending position is current position i. so possible subarray num: min(jmin, jmax)-(jbad+1)+1 =  min(jmin, jmax) - jbad.
    it is possible that range [jbad+1, min(jmin, jmax)] is not valid, hence do a max(0, min(jmin, jmax) - jbad) when updating the count
    for example an invalid [1,7,5] minK 1, maxK 5
    """
    res = 0
    jmin, jmax, jbad = -1, -1, -1 # initialize the index
    for ind, val in enumerate(nums):
        # update the 3 index accordingly
        if not (minK <= val <= maxK): jbad = ind
        if val == minK: jmin = ind
        if val == maxK: jmax = ind
        res += max(0, min(jmax, jmin) - jbad)
    return res

In [9]:
countSubarrays([1,3,5,2,7,5], 1, 5)

2

<b> 408 valid word abbreviation </b>

loop over word and abbr for comparison, tricky part is how to update the index i accordingly for comparison

In [11]:

def validWordAbbreviation(word: str, abbr: str) -> bool:
    """
    loop over each char in abbr, if it is char, compare with current char in word at position i,
    if it is number, keep track the value
    update the i accordingly
    """
    i = 0
    n = len(word)
    num = 0
    for c in abbr:
        if c.isalpha():
            i += num
            num = 0
            if i>=n or word[i] != c: return False
            i +=1
        elif c.isnumeric():
            if num == 0 and c == "0": return False
            num = num*10 + int(c)
    if num > 0:
        # for case that last few chars in abbr are number
        return (i+num) == n
    else:
        return i == n

<b> 921 minimum add to make parenthesis valid </b>

each ( should have a pairing right ) in order, vise versa. my intuitive is to use a stack to save the (, loop over the string, if see a ), pop out the ( in stack if the stack is not empty, however, it can be optimized to use an integer to keep the value of ( encoutered, no need to store it in a stack and do pop.

In [12]:

def minAddToMakeValid(s: str) -> int:
    """
    using stack, one ( must pair with one ) in order

    """
    count_of_left_p = 0 # only store (
    res = 0
    for c in s:
        if c == "(":
            count_of_left_p +=1
        else:
            # if left ( is available, the ) has a pair, otherwise it is an invalid ) and need to add a (
            if count_of_left_p > 0:
                count_of_left_p -= 1
            else:
                res += 1
    # it can happen that the ( has no pairing )
    res += count_of_left_p
    return res

<b> 1004 max consecutive ones III </b>

smart to use sliding windows. no need to use another res var do res = max(res, j-i) because the sliding window will stop at the largest window it has ever encoutered, even though the later window contains more 0 than allowed, the max length is valid because it previously saw such a window with such length that is valid.

In [14]:

def longestOnes(nums: List[int], k: int) -> int:
    """
    the problem is transformed into:
    find the longest subarray with at most K 0s
    using sliding window: 
    - if num_of_zero <= K, increment j
    - if num_of_zero > K, increment i and j
    """
    i = 0
    # no need to have another res var to keep track of the longest subarray length, the i and j sliding with the limit of k will find its longest subarray at the end
    for j in range(len(nums)):
        # one zero is found
        if nums[j] == 0: k-=1
        # no quota left, moving i potentially increase k
        if k < 0:
            k += 1-nums[i] # k may or may not increase
            i+=1
    return j - i+1


<b> follow up problems: what if the nums is an infinite stream </b>

my thoughts: store the index of value 0 along the way when processing the stream, everytime when k < 0 and try to update index i, check if i is in the recorded index list (recorded_list[0]), if yes, then k +=1, i +=1, and remove the first item from the recorded list. otherwise it means index i has value 1.

<b> 424 longest repeating character replacement </b>

still a bit hard to understand. try to use the similar pattern as above
- the count_map stores the count number for each distinct char from string s
- max_freq tracks the max frequency of char encountered. however it does not decrease during the update
- j - i + 1 is the substring length, max_freq is the lengh of the most frequnetly appeared char within this window. j-i+1 - max_freq is the number of other char that needs to be replaced, it has the quota k.


the reason of why max_freq no need to update after i is shrinking
- it tracks the max freq ever seen, which is related to the max length of substring that can be created.
- after find the max length of result, why shrink it, just keep moving the j and i until find next larger value to update the final result.

In [17]:
from collections import defaultdict

def characterReplacement(self, s: str, k: int) -> int:
    i = 0 # left pointer
    count_map = defaultdict(int) # store the {char: count}
    res = 0
    max_freq = 0 # keep the max count of the char encountered so far
    for j in range(len(s)):
        # update the count map
        count_map[s[j]] += 1
        max_freq = max(max_freq, count_map[s[j]])
        # check the current window size - most frequent char count, which is the num of chars that should be flipped.
        if (j - i + 1) - max_freq <= k:
            # valid case, update the res
            res = max(res, j-i+1)
        else:
            # invalid case, k is out of quota
            # s[i] is a char, it can appear later in s
            # its count is updated
            count_map[s[i]] -= 1 # <- this part, why after updating the count of s[i] no need to update the max_freq
            i += 1
    return res


an easier way to understand is to replace the max_freq using max() function, but it takes O(26) instead of O(1)

In [18]:
from collections import defaultdict

def characterReplacement(s: str, k: int) -> int:
    i = 0 # left pointer
    count_map = defaultdict(int) # store the {char: count}
    res = 0
    max_freq = 0 # keep the max count of the char encountered so far
    for j in range(len(s)):
        # update the count map
        count_map[s[j]] += 1
        # this max function takes O(N)
        max_freq = max(count_map.values())
        # check the current window size - most frequent char count, which is the num of chars that should be flipped.
        if (j - i + 1) - max_freq <= k:
            # valid case, update the res
            res = max(res, j-i+1)
        else:
            # invalid case, k is out of quota
            # s[i] is a char, it can appear later in s
            # its count is updated
            count_map[s[i]] -= 1
            i += 1
    return res


<b> 2401 longest nice subarray </b>

using bit operation, a bit difficult to understand at the beginning.  
- &: only 1 & 1 = 1, to have bitwise AND of pair of elements equal to 0, means not too bit position from the two numbers can be 1 at the same time
- ^: when two bit value are different then it is 1, so it is used to offsetting a value 
- |: as long as one bit position from two numbers is 1 the result is 1, so it can be used to add values

the idea is same as all other sliding window problem, when the condition met, keep enlarging the window size by incrementing value j, and at the same time updating the temp var AND to include the current value. When the condition break, shrink the window by incrementing value i, and remove the num[i] from the temp value AND.


In [19]:
def longestNiceSubarray(nums: List[int]) -> int:
    # AND: check all values in the current window
    # i: left windown position
    res = AND = i = 0
    for j in range(len(nums)):
        # when AND & nums[j] == 0, 
        # j can be in the nice subarray
        # otherwise window shrink by ++i, and the ith element
        # need to be removed out from AND
        while AND & nums[j]:
            # ^ operation can remove the item by offsetting
            AND = AND ^ nums[i]
            i += 1
        # add item nums[j] to the AND var by | operation
        AND = AND | nums[j]
        res = max(res, j-i+1)
    return res


<b> 2461 maximum sum of distinct subarrays with length k</b>

using a set to track the visited item to ensure the subarray only contains distinct value.  
check subarray length before updating the res variable, to ensure covering the edge case which is that no satisfied subarray exist.  

similar to other sliding window problems, the visited_set and the tmp variables are used to keep track of the status, when condition met, moving forward, when condition not met (item saw before, or window size exceed the limit), moving the left side of the window.

In [20]:

def maximumSubarraySum(nums: List[int], k: int) -> int:
    i = 0
    res = 0
    tmp = 0 # temp sum of subarray in the window
    # how to know if element in the visited subwarray is distinct
    visited_set = set()
    for j in range(len(nums)):
        while nums[j] in visited_set:
            # shrink window
            visited_set.remove(nums[i])
            tmp -= nums[i]
            i += 1

        visited_set.add(nums[j])
        tmp += nums[j]
        if j-i+1 > k:
            # move window
            visited_set.remove(nums[i])
            tmp -= nums[i]
            i += 1
        # edge case, it could happen that cannot find such a subarray
        if j-i+1 == k:
            res = max(res, tmp)
    return res 


<b> 2337 move pieces to obtain string </b>

very smart, the solution is not intuitive at the beginning.

the first while loop should check i<=n and j<=n instead of i<n and j<n otherwise the test case "_L" "LR" wont pass

In [1]:

def canChange(start: str, target: str) -> bool:
    n = len(start)
    i = j = 0
    # should be i<=n and j<=n instead of i<n and j<n otherwise the test case "_L" "LR" wont pass
    while i <= n and j <= n:
        while i<n and start[i] == "_": i+=1
        while j<n and target[j] == "_": j+=1
        if i == n or j == n:
            # both should all reach the end otherwise there is one string that still have non empty char left
            return i==n and j==n
        # the non empty char should match in order
        if start[i]!=target[j]: return False
        if start[i]=="L":
            # target[j] is also L, but its left should have more empty space if j >i, however, start string does not have that much empty space that can achieve it
            if j > i: return False
        else:
            # start[i] and target[j] all R, but start[i] can only move right, it cannot move to a previous location to satisfy the criteria
            if j < i: return False
        i+=1
        j+=1
    return True

In [2]:
canChange("_L", "LR")

False

<b> 3404 count special subsequences </b>

very smart way to use the indexes. A[p]*A[r] = A[q]*A[s] -> A[p]/A[q] = A[s]/A[r]
where p < q < r < s.

first fix the value of q and r, then first find all possible combination of p q, saving the result of nums[p]/nums[q] to hashmap, then find all possible combination of r and s, calculting the vlaue of nums[r]/nums[s] and see if there is any mathing pair from the hashmaps.

q and r in each loop is fixed, it does not mean the q and r aleays differ in 2, it just a way to help find all possible combination of p and q.

time complexity:
-   O(N) for the first for loop of r
-   nested for loop of p is O(r), for loop of s is O(n-r), so nested loop in total is O(N)
-   in total two loop takes O(N*N)

space complexity:
-   hashmap O(N)

In [3]:
from collections import defaultdict

def numberOfSubsequences(self, nums: List[int]) -> int:
    """
    A[p]*A[r] = A[q]*A[s] -> A[p]/A[q] = A[s]/A[r]
    p < q < r < s
    """
    n = len(nums)
    res = 0
    count = defaultdict(int)
    #  p < q < r < s, r can only starting at index 4
    # fixing position of q and r, then loop through the rest of array to find the possible p, then s
    for r in range(4, n-2):
        # why fix q: q is used to calculate all possible p,q pair
        # the pair of pq working with sr may not come from current loop, but definitely come from all the combinations that have encoutered previously which are saved in the hashmap 
        q = r-2
        # q-1, r+2 because at least seperating by 1 element
        for p in range(q-1):
            count[nums[p]*1.0/nums[q]] += 1
        for s in range(r+2, n):
            res += count[nums[s]*1.0/nums[r]]   
    return res


<b> 149 Max points on a line </b>

Given an array of points where points[i] = [xi, yi] represents a point on the X-Y plane, return the maximum number of points that lie on the same straight line.

the brute force way of looping over all possible N(N+1)/2 does not take care of the edge cases properlly

special cases:
-   two line in parallel, so the dx/dy is the same, but the dots from two lines are really from two lines. so simply keeping one hashmap to keep track of dx/dy is not good.
-   also using dx/dy which is a double value as key in hashmap is not appropriate.
-   there can be duplicated points


here the findGCD method helps to find the proper (dx, dy) tuple as the key to the hashmap. also when fix a, then find b and keep the (dx, dy) value can avoid the case of finding nodes from two parallel lines.



The following may not be true:
        # for parallel line like y=ax+b1, y=ax+b2, even though two lines having same a
        # as slope, which is the (dx, dy) key used in counter_map, the b is different so should be two diff lines
        # when loop in this way (fix one point, and loop the rest points), it is not likely to add poits with same a but diff b into the same counter map

In [7]:
from collections import defaultdict

def maxPoints(points):
    """
    :type points: List[List[int]]
    :rtype: int
    """
    # 2 points form a line, so total number of possible line is no greater than C(n, 2)
    # where n = len(points)
    # the brute force way is to loop over all possible N(N+1)/2 combination,  get the (a,b) of the line
    # in a map to keep check of its count, and return the count
    def findGCD(dx, dy):
        if dy == 0: return dx
        else: return findGCD(dy, dx%dy)

    n = len(points)
    if n<=1:
        return 1
    res = 0
    for i in range(n):
        # why keep a counter_map per i?
        # suppose [a,b,c] in a line
        # for i = a, the res we get is 3, it already check [a,b], [a,c]
        # when i = b, the [b,c] point is already included in the previous line

        counter_map = defaultdict(int) # each i has one, to elimiate parallel line
        dup = 1
        for j in range(i+1, n):
            dx = points[i][0] - points[j][0]
            dy = points[i][1] - points[j][1]
            if dx == 0 and dy==0:
                # duplicated nodes
                dup +=1
            else:
                g = findGCD(dx, dy)
                counter_map[(dx/g, dy/g)] += 1
        res = max(res, dup)
        for p in counter_map:
            # plus 1 to get the right count of points num
            res = max(res, counter_map[p]+1)
    return res

In [7]:
def findGCD(dx, dy):
    if dy == 0: return dx
    else: return findGCD(dy, dx%dy)

In [8]:
findGCD(0, 3)

3

In [9]:
findGCD(3, 0)

3

<b> 2762 countinous subarrays </b>

when moving the left and right pointers, while the max and min value within this window does not exceed the limit, this windown will create r-l+1 valid subarrays with end position at r, and start position from l to r with r included.

In [4]:
from collections import Counter

def continuousSubarrays(nums: List[int]) -> int:
    """
    valid count for subarray from i to j: j-i+1
    because every subarray with starts between [i,j] including j and ends at j is considered valid, so actually the length of i to j
    for example [i1, i2, i3, i4]: [i1 to i4, i2 to i4, i3 to i4, i4 to i4], for the subarray like i1 to i2 is already counted when the end index is i2 so not duplicated here.

    time complexity:
    - O(N) loop over r from 0 to n
    - O(N), inside while loop, max and min takes O(K) k is the window size visited now, since the max - min <=2, K is bounded by 3
    - so in total O(N*K) => O(N)
    """
    n = len(nums)
    freq = defaultdict(int)
    l = r = 0
    count = 0
    while r < n:
        freq[nums[r]] += 1
        while max(freq)- min(freq) > 2:
            
            # shrink the left pointer
            freq[nums[l]]-=1
            if freq[nums[l]]==0:
                del freq[nums[l]]
            l += 1
        count += (r - l + 1)
        r += 1
    return count
    

instead of using hashmap, we can use 2 deque, but i still think the hashmap way is easier for understanding

In [6]:
from collections import defaultdict


def continuousSubarrays(nums: List[int]) -> int:
    """
    valid count for subarray from i to j: j-i+1
    because every subarray with starts between [i,j] including j and ends at j is considered valid, so actually the length of i to j
    for example [i1, i2, i3, i4]: [i1 to i4, i2 to i4, i3 to i4, i4 to i4], for the subarray like i1 to i2 is already counted when the end index is i2 so not duplicated here.

    sliding window way, then using two pq to track the min and max value encountered.

    time complexity:
    - O(N) for while loop
    - O(N) for the left pointer movement because it only move forward once 
    - total O(N)
    
    """
    min_q = deque() # i for nums[i] increasing monotonic
    max_q = deque() # i for nums[i] decreasing monotonic
    l = 0
    count = 0
    for r, num in enumerate(nums):
        # max_q keep decreasing
        while max_q and nums[max_q[-1]] < num:
            max_q.pop()
        max_q.append(r)
        # min_q keep increasing
        while min_q and nums[min_q[-1]] > num:
            min_q.pop()
        min_q.append(r)

        """
        max_q[0]: ind of the largeset value encountered right now
        min_q[0]: ind of the smallest value encountered right now
        directly update l to the right position, and check again
        so it is quicker and only takes O(N) for the whole operation
        because left only move forward once for the entire for loop
        """
        while max_q and min_q and nums[max_q[0]] - nums[min_q[0]] > 2:
            # shrink window size
            if max_q[0] < min_q[0]:
                l = max_q[0]+1
                max_q.popleft()
            else:
                l = min_q[0] + 1
                min_q.popleft()
        count += (r - l)+1
    return count